In [ ]:
# === Held-out evaluation for latest v2 run (macro/micro Dice, CSV+JSON) ===
from pathlib import Path
import importlib.util, json, time
import numpy as np

# --------- Paths you can tweak ----------
RUN_DIR   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356")
TEST_DIR  = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_lores")
T1_DIR    = TEST_DIR / "t1"
MSK_DIR   = TEST_DIR / "masks"
TRAIN_MOD = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
# ---------------------------------------

# ---- Import training module (gives us loader/preproc utils + custom layers) ----
spec = importlib.util.spec_from_file_location("arc_seg_train", TRAIN_MOD)
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

# ---- Find model to load (prefer full .keras; else build and load best weights) ----
models_dir     = RUN_DIR / "models"
callbacks_dir  = RUN_DIR / "callbacks"
full_models    = sorted(models_dir.glob("*.keras"))
best_weights   = callbacks_dir / "best_model_dynamic.weights.h5"
cfg_json_path  = models_dir / "config.json"  # written by the training code

model = None
INPUT_SHAPE = None

# custom_objects for loading .keras
custom_objects = {
    "ResidualConvBlock": seg.ResidualConvBlock,
    "VisionMambaBlock": seg.VisionMambaBlock,
    "SAM2Attention": seg.SAM2Attention,
    "CombinedLoss": seg.CombinedLoss,
    "dice_coefficient": seg.dice_coefficient,
    "dice_loss": seg.dice_loss,
    "boundary_loss": seg.boundary_loss,
}

try:
    from keras.saving import load_model as keras_load_model
except Exception:
    from tensorflow.keras.models import load_model as keras_load_model

if full_models:
    model_path = full_models[-1]
    print(f"Loading FULL model: {model_path}")
    model = keras_load_model(model_path, compile=False, custom_objects=custom_objects)
    INPUT_SHAPE = tuple(model.input_shape[1:])
else:
    # Build a matching model from saved config.json, then load weights
    assert cfg_json_path.exists(), f"Missing {cfg_json_path}"
    with open(cfg_json_path) as f:
        saved_cfg = json.load(f)
    # Minimal fields needed to rebuild the same shapes
    cfg = seg.DynamicTrainingConfig(
        DATA_DIR=TEST_DIR,  # dummy root; we won't train
        MODEL_DIR=models_dir,
        CALLBACKS_DIR=callbacks_dir,
        INPUT_SHAPE=tuple(saved_cfg["INPUT_SHAPE"]) if saved_cfg.get("INPUT_SHAPE") else None,
        BASE_FILTERS=int(saved_cfg.get("BASE_FILTERS", 8)),
        SAM_HEADS=int(saved_cfg.get("SAM_HEADS", 2)),
    )
    if cfg.INPUT_SHAPE in (None, (), []):
        raise RuntimeError("INPUT_SHAPE missing in saved config; cannot rebuild model.")
    INPUT_SHAPE = cfg.INPUT_SHAPE
    print("Rebuilding model from config.json and loading best weights…")
    model = seg.build_dynamic_model(cfg)
    model.load_weights(str(best_weights))

print("INPUT_SHAPE:", INPUT_SHAPE)

# ---- Build the held-out list (use separated subfolders to avoid duplicates) ----
if T1_DIR.exists() and MSK_DIR.exists():
    cfg_eval = seg.DynamicTrainingConfig(DATA_DIR=TEST_DIR, IMAGES_DIR=T1_DIR, MASKS_DIR=MSK_DIR,
                                         MODEL_DIR=RUN_DIR/"_tmp_models", CALLBACKS_DIR=RUN_DIR/"_tmp_callbacks")
else:
    cfg_eval = seg.DynamicTrainingConfig(DATA_DIR=TEST_DIR,
                                         MODEL_DIR=RUN_DIR/"_tmp_models", CALLBACKS_DIR=RUN_DIR/"_tmp_callbacks")

cfg_eval.INPUT_SHAPE = INPUT_SHAPE
pairs, lesion_presence = seg.load_generic_dataset(cfg_eval)
print(f"Pairs: {len(pairs)} | % non-empty masks: {lesion_presence.mean()*100:.1f}%")

# ---- Dice helpers ----
def dice_soft(y, p):
    y = y.astype(np.float64); p = p.astype(np.float64)
    inter = (y * p).sum()
    return (2.0*inter) / (y.sum() + p.sum() + 1e-12)

def dice_hard(y, p, th=0.5):
    pb = (p >= th).astype(np.float64)
    inter = (y*pb).sum()
    return (2.0*inter) / (y.sum() + pb.sum() + 1e-12)

# ---- Evaluate (macro: per-case mean; micro: global) ----
macro_softs, macro_hards = [], []
s_inter_soft = np.float64(0.0)
s_sumy_soft  = np.float64(0.0)
s_sump_soft  = np.float64(0.0)

TH = 0.50  # you can sweep later

for i, (img_p, msk_p) in enumerate(pairs, 1):
    img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1]).astype(np.float32)
    y   = seg._load_and_preprocess_mask(str(msk_p),  INPUT_SHAPE[:-1]).astype(np.float32)

    x = np.zeros((1,*INPUT_SHAPE), np.float32)
    x[0,...,0] = img
    p = model.predict(x, verbose=0)[0,...,0].astype(np.float32)

    # per-case dice
    ds = dice_soft(y, p)
    dh = dice_hard(y, p, th=TH)
    macro_softs.append(ds); macro_hards.append(dh)

    # micro-soft accumulators
    s_inter_soft += (y.astype(np.float64) * p.astype(np.float64)).sum()
    s_sumy_soft  += y.sum(dtype=np.float64)
    s_sump_soft  += p.sum(dtype=np.float64)

    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] last case soft={ds:.4f} hard@{TH:.2f}={dh:.4f}")

macro_soft = float(np.mean(macro_softs))
macro_hard = float(np.mean(macro_hards))
micro_soft = float((2.0*s_inter_soft) / (s_sumy_soft + s_sump_soft + 1e-12))

# compute micro-hard at the same threshold
# we need a second pass for exact global hard (memory-safe). Do a quick second pass:
s_inter_hard = np.float64(0.0)
s_sumy_hard  = np.float64(0.0)
s_sump_hard  = np.float64(0.0)
for img_p, msk_p in pairs:
    y = seg._load_and_preprocess_mask(str(msk_p), INPUT_SHAPE[:-1]).astype(np.float32)
    x = np.zeros((1,*INPUT_SHAPE), np.float32)
    x[0,...,0] = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1]).astype(np.float32)
    p = model.predict(x, verbose=0)[0,...,0]
    pb = (p >= TH).astype(np.float64)
    s_inter_hard += (y.astype(np.float64) * pb).sum()
    s_sumy_hard  += y.sum(dtype=np.float64)
    s_sump_hard  += pb.sum(dtype=np.float64)

micro_hard = float((2.0*s_inter_hard) / (s_sumy_hard + s_sump_hard + 1e-12))

# ---- Report + save ----
print("\n=== HELD-OUT RESULTS ===")
print(f"Per-case (macro) soft Dice      : {macro_soft:.4f}")
print(f"Per-case (macro) hard Dice @ {TH:.2f}: {macro_hard:.4f}")
print(f"Global (micro) soft Dice        : {micro_soft:.4f}")
print(f"Global (micro) hard Dice @ {TH:.2f}  : {micro_hard:.4f}")
print(f"Val set size: {len(pairs)} cases")

# Save into this run folder (so results travel with the model)
out_dir = RUN_DIR / "test_eval"
out_dir.mkdir(parents=True, exist_ok=True)
ts = time.strftime("%Y%m%d_%H%M%S")

# per-case CSV
csv_path = out_dir / f"test_metrics_{ts}.csv"
with open(csv_path, "w") as f:
    f.write("case,soft_dice,hard_dice_at_{:.2f}\n".format(TH))
    for (img_p, _), ds, dh in zip(pairs, macro_softs, macro_hards):
        f.write(f"{img_p.stem},{ds:.6f},{dh:.6f}\n")

# summary JSON
summary = {
    "threshold": TH,
    "macro_soft": macro_soft,
    "macro_hard": macro_hard,
    "micro_soft": micro_soft,
    "micro_hard": micro_hard,
    "n_cases": len(pairs),
    "run_dir": str(RUN_DIR),
}
json_path = out_dir / f"test_metrics_summary_{ts}.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nWrote per-case CSV ->", csv_path)
print("Wrote summary JSON ->", json_path)
